In [ ]:
import lightning as L
import timm
import torch
from pytorch_metric_learning import miners
import albumentations as A

In [1]:
class CarsEmbedder(L.LightningModule):
    def __init__(self, margin=1.0, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name='efficientnet_b3', pretrained=True, num_classes=0)

        self.loss = torch.nn.TripletMarginLoss(margin=margin)
        self.miner = miners.TripletMarginMiner(margin=margin, type_of_triplets="semihard")

        self.validation_step_outputs = []
        self.validation_step_labels = []

    def forward(self, imgs):
        return self.model(imgs)

    def training_step(self, batch):
        imgs, labels = batch

        embeds = self(imgs)

        anchors_indices, positives_indices, negatives_indices = self.miner(embeds, labels)
        loss = self.loss(embeds[anchors_indices], embeds[positives_indices], embeds[negatives_indices])
        return loss

    @staticmethod
    def precision_k(embeds, labels, k=5):
        res = 0
        for i, embed in enumerate(embeds):
            distances = []
            main_label = labels[i]
            for j, other_embeds in enumerate(embeds):
                distance = torch.nn.functional.cosine_similarity(embed, other_embeds)
                distances.append((j, distance))
            nearest = sorted(distances, key=lambda x: x[1], reverse=True)[1:k+1]
            res += sum(1 for idx, dist in nearest if labels[idx] == main_label and dist!=1.0) / k
        return res / len(embeds)

    @staticmethod
    def recall_k(embeds, labels, k=5):
        res = 0
        for i, embed in enumerate(embeds):
            distances = []
            main_label = labels[i]
            main_label_count = sum(1 for label in labels if label == main_label)
            for j, other_embeds in enumerate(embeds):
                distance = torch.nn.functional.cosine_similarity(embed, other_embeds)
                distances.append((j, distance))
            nearest = sorted(distances, key=lambda x: x[1], reverse=True)[1:k+1]
            res += sum(1 for idx, dist in nearest if labels[idx] == main_label) / main_label_count
        return res / len(embeds)

    @staticmethod
    def m_ap(embeds, labels):
        res = 0
        for i, embed in enumerate(embeds):
            distances = []
            main_label = labels[i]
            for j, other_embed in enumerate(embeds):
                distance = torch.nn.functional.cosine_similarity(embed, other_embed)
                distances.append((j, distance))
            ranked = sorted(distances, key=lambda x: x[1], reverse=True)[1:]

            relevants_count = sum(1 for label in labels if label == main_label)
            res += sum(CarsEmbedder.precision_k(embeds, labels, k) for k in range(1, len(ranked)+1) if labels[ranked[k-1][0]] == main_label) / relevants_count
        return res / len(embeds)

    def validation_step(self, batch):
        imgs, labels = batch

        embeds = self.forward(imgs)

        anchors_indices, positives_indices, negatives_indices = self.miner(embeds, labels)
        loss = self.loss(embeds[anchors_indices], embeds[positives_indices], embeds[negatives_indices])
        self.log("val_loss", loss)

        self.validation_step_outputs.append(embeds.detach().cpu())
        self.validation_step_labels.append(labels.detach().cpu())

    def on_validation_epoch_end(self):

        embeds = torch.cat(self.validation_step_outputs, dim=0)
        labels = torch.cat(self.validation_step_labels, dim=0)

        prec_k = self.precision_k(embeds, labels)
        rec_k = self.recall_k(embeds, labels)
        m_ap = self.m_ap(embeds, labels)
        self.log(f"val_precision@k", prec_k)
        self.log(f"val_recall@k", rec_k)
        self.log("val_mAP", m_ap)

        self.validation_step_labels.clear()
        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return optimizer

IndentationError: expected an indented block after function definition on line 22 (2323487646.py, line 25)